In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt


def interpolate_dataframe(df, x_values):
    """
    Linearly interpolate y-values for arbitrary x-values.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing columns 'x' and 'y'.
    x_values : array-like
        x-values at which to interpolate.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns 'x' and 'y', where 'y' contains the
        interpolated values.
    """

    xColumn = df.columns[0]
    yColumn = df.columns[1]

    # Ensure data are sorted by x
    df_sorted = df.sort_values(xColumn)

    x = df_sorted[xColumn].to_numpy()
    y = df_sorted[yColumn].to_numpy()

    x_values = np.asarray(x_values)

    # Linear interpolation
    y_interp = np.interp(x_values, x, y)

    return pd.DataFrame({
        xColumn: x_values,
        yColumn: y_interp
    })

def fittingError(dfGroundTruth, dfSimulated, xRange=[0,2.5]):
    xArray = np.linspace(xRange[0],xRange[1], 100) # just interpolate to high number of points to approximate integral via sum. 
    dfGroundTruthInterpolated = interpolate_dataframe(dfGroundTruth, xArray)
    dfSimulatedInterpolated = interpolate_dataframe(dfSimulated, xArray)

    df = dfGroundTruthInterpolated
    df.rename(columns={df.columns[0]:"nominalStrain", df.columns[1]:"stressGroundTruth"}, inplace=True)
    df["stressSimulated"] = dfSimulatedInterpolated[dfSimulatedInterpolated.columns[1]]

    df["stressDelta"] = df["stressGroundTruth"] - df["stressSimulated"]
    sumOfDeltas = np.sum(np.abs(df["stressDelta"]))
    sumOfStressGroundTruth = np.sum(df["stressGroundTruth"])

    return float(sumOfDeltas/sumOfStressGroundTruth)


# Load Data

In [ ]:

tpu1GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU1.csv") 
tpu2GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU2.csv")
tpu3GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU3.csv")

with open('results_h3_linesearchNone/niklasTesting2StressStrain.pickle', 'rb') as f:
    stressStrainSimulatedTPU1 = pickle.load(f)

with open('results_h3_tpu3/niklasTesting2StressStrain.pickle', 'rb') as f:
    stressStrainSimulatedTPU3 = pickle.load(f)

with open('results_hexPaperLike_tpu3/niklasTesting2StressStrain.pickle', 'rb') as f:
    stressStrainSimulatedTPU3 = pickle.load(f)

In [ ]:
with open('results/niklasTesting2StressStrain.pickle', 'rb') as f:
    stressStrainSimulatedTPU1 = pickle.load(f)

# Plots

In [ ]:

fig, ax = plt.subplots(1,1, figsize=(9,4))

ax.plot(tpu1GroundTruth["nominalStrain"], tpu1GroundTruth[" nominalStressInMPa"], label="TPU1 experimental data")
# ax.plot(tpu2["nominalStrain"], tpu2[" nominalStressInMPa"], label="TPU2 experimental data")
# ax.plot(tpu3["nominalStrain"], tpu3[" nominalStressInMPa"], label="TPU3 experimental data")
ax.plot(stressStrainSimulatedTPU1["nominalStrain"], stressStrainSimulatedTPU1["nominalStress"], label="TPU1 simulated data")
# ax.plot(stressStrainSimulatedTPU3["nominalStrain"], stressStrainSimulatedTPU3["nominalStress"], label="TPU3 simulated data")

ax.legend()
ax.set_title(f"Stress Strain Curve")
plt.show()

In [ ]:
maxStrainContainedInBothDatasets = np.min([np.max(tpu1GroundTruth["nominalStrain"]), np.max(stressStrainSimulatedTPU1["nominalStrain"])])
print(f"error score up to {maxStrainContainedInBothDatasets} max strain")
err = fittingError(tpu1GroundTruth, stressStrainSimulatedTPU1, xRange=[0,maxStrainContainedInBothDatasets])
print(f"error score: {err}")